# Recurrent Neural Network
###### Addition using tensorflow

Import dependencies

In [1]:
import numpy as np
import tensorflow as tf

### Pre-process data

In [2]:
# seed random number generators
rng = np.random.default_rng(seed=42)
tf.random.set_seed(42)

In [3]:
total_test_cases = 1000
train_test_ratio = 0.8
disp_test = 20

tmp_list = []
features = []
labels = []

function to reverse bits of a number

In [4]:
def reverse_bits(n, n_bits):
    rev = 0
    for i in range(n_bits):
        if n & (1 << i):
            rev |= 1 << ((n_bits - 1) - i)
    return rev

generate data (addition)

In [5]:
for i in range(total_test_cases):
    a = rng.integers(0, 128)
    b = rng.integers(0, 128)
    c = a + b

    a = reverse_bits(a, 8)
    b = reverse_bits(b, 8)
    c = reverse_bits(c, 8)

    features.append([a, b])
    labels.append(c)

features = np.array(features, dtype=np.uint8).reshape(-1, 1)
labels = np.array(labels, dtype=np.uint8).reshape(-1, 1)
features = np.unpackbits(features, axis=1)
labels = np.unpackbits(labels, axis=1)

labels = np.expand_dims(labels, 2)

for i in range(len(labels)):
    tmp_list.append([features[2 * i], features[2 * i + 1]])

features = np.array(tmp_list)

split into train-test set and transpose the array

In [6]:
features_train = np.transpose(features[:int(train_test_ratio * len(features))],
                              [2, 0, 1]).astype(np.float32)
features_test = np.transpose(features[int(train_test_ratio * len(features)):],
                             [2, 0, 1]).astype(np.float32)

labels_train = np.transpose(labels[:int(train_test_ratio * len(labels))],
                            [1, 0, 2]).astype(np.float32)
labels_test = np.transpose(labels[int(train_test_ratio * len(labels)):],
                           [1, 0, 2]).astype(np.float32)

## Neural Network

hyper-parameters

In [7]:
n_input_neurons = 2
n_rnn_neurons = 12
n_output_neurons = 1
sequence_len = 8

learning_rate = 0.01

n_epochs = 100

weights and biases

In [8]:
layer_op = {
    'weight': tf.Variable(tf.random.normal([n_rnn_neurons, n_output_neurons], stddev=0.1)),
    'bias': tf.Variable(tf.random.normal([n_output_neurons], stddev=0.1))
}

#### Model

In [9]:
rnn = tf.keras.layers.SimpleRNN(n_rnn_neurons, return_sequences=True)


def model(X):
    # [sequence_len, batch_size, n_input_neurons] -> [batch_size, sequence_len, n_input_neurons]
    x_batch = tf.transpose(X, [1, 0, 2])
    rnn_ops = tf.transpose(rnn(x_batch), [1, 0, 2])

    pred_op = tf.map_fn(
        lambda x:
            tf.nn.sigmoid(tf.matmul(x, layer_op['weight']) + layer_op['bias']),
        rnn_ops)
    return pred_op

#### Error and Optimizer

In [10]:
# adam-optimizer
optimizer = tf.optimizers.Adam(learning_rate)


@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        pred_op = model(x)
        # mean-squared error
        error = tf.reduce_mean(0.5 * tf.square(pred_op - y))
    trainable_variables = rnn.trainable_variables + [
        layer_op['weight'], layer_op['bias']
    ]
    grads = tape.gradient(error, trainable_variables)
    optimizer.apply_gradients(zip(grads, trainable_variables))
    return error

#### Start Training

In [11]:
print("########## TRAIN ##########")

# Epoch training
for epoch in range(n_epochs):
    err = train_step(features_train, labels_train)

    if epoch % 10 == 0:
        print(f"Epoch: {epoch} Error: {float(err):f}")

print("\n########## TEST ##########")

pred_op = model(features_test).numpy()
op = np.transpose((pred_op > 0.5).astype(np.uint8), [1, 0, 2])
op = np.packbits(op)
op = [reverse_bits(x, 8) for x in op]

c = np.transpose(labels_test.astype(np.uint8), [1, 0, 2])
c = np.packbits(c)
c = [reverse_bits(x, 8) for x in c]

a = features_test[:, :, 0]
a = np.transpose(a.astype(np.uint8), [1, 0])
a = np.packbits(a)
a = [reverse_bits(x, 8) for x in a]

b = features_test[:, :, 1]
b = np.transpose(b.astype(np.uint8), [1, 0])
b = np.packbits(b)
b = [reverse_bits(x, 8) for x in b]

success = 0

for i in range(len(op)):

    if c[i] == op[i]:
        success += 1

    if i < disp_test:
        print(f"{a[i]} + {b[i]} = {c[i]} \t --> {str(c[i] == op[i]):>5} ")

print(f"\nSuccess: {success}/{len(op)}, Accuracy = {success / len(op) * 100:f}")

########## TRAIN ##########
Epoch: 0 Error: 0.125578
Epoch: 10 Error: 0.122905
Epoch: 20 Error: 0.121175
Epoch: 30 Error: 0.117696
Epoch: 40 Error: 0.110733
Epoch: 50 Error: 0.097301
Epoch: 60 Error: 0.076750
Epoch: 70 Error: 0.051579
Epoch: 80 Error: 0.030769
Epoch: 90 Error: 0.017215

########## TEST ##########
38 + 94 = 132 	 -->  True 
105 + 40 = 145 	 -->  True 
15 + 113 = 128 	 -->  True 
18 + 76 = 94 	 -->  True 
30 + 16 = 46 	 -->  True 
99 + 18 = 117 	 -->  True 
85 + 88 = 173 	 -->  True 
7 + 22 = 29 	 -->  True 
101 + 64 = 165 	 -->  True 
15 + 126 = 141 	 -->  True 
33 + 0 = 33 	 -->  True 
118 + 2 = 120 	 -->  True 
70 + 127 = 197 	 -->  True 
73 + 74 = 147 	 -->  True 
49 + 16 = 65 	 -->  True 
73 + 114 = 187 	 -->  True 
11 + 112 = 123 	 -->  True 
53 + 68 = 121 	 -->  True 
21 + 79 = 100 	 -->  True 
23 + 34 = 57 	 -->  True 

Success: 200/200, Accuracy = 100.000000
